# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdrayan001/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will use two supervised models for this ranking task:

- **Logistic Regression** as the simple, readable model.
- **Random Forest** as the stronger non-linear model.

The target is the current decline proxy: `trend_direction == "down"`.

Because the business question is "which pages should be reviewed first?", I will rank pages using each model's predicted probability of the decline proxy and evaluate the ranking with Precision@50.

I will not use `trend_direction`, `trend_pct`, content/client IDs, or existing product decision flags as features. The model is decision-support for review prioritization, not a prediction of Google's algorithm or a causal claim about refresh results.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Load the same starter dataset used for ML-07
# ---------------------------------------------------------

paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next(p for p in paths if p.exists())
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

# ---------------------------------------------------------
# Target / proxy
# ---------------------------------------------------------

df["declining_proxy"] = (
    df["trend_direction"] == "down"
).astype(int)

# ---------------------------------------------------------
# Features
# ---------------------------------------------------------
# These are observable page-level signals.
# IDs are grouping fields only, not model features.

feature_columns = [
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
]

print("\nFeatures:")
for feature in feature_columns:
    print("-", feature)

print("\nTarget distribution:")
print(df["declining_proxy"].value_counts().rename(
    index={0: "not_down", 1: "down"}
))

print(
    "\nDecline proxy rate:",
    round(df["declining_proxy"].mean(), 3)
)

# ---------------------------------------------------------
# Leakage guard
# ---------------------------------------------------------

forbidden_fields = [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
    "is_declining_label",
    "needs_ctr_fix",
    "is_quick_win",
    "needs_engagement_fix",
    "is_underperformer",
    "health_score",
]

leaked_features = [
    field for field in forbidden_fields
    if field in feature_columns
]

print("\nForbidden fields used as features:", leaked_features)

assert leaked_features == []

print("Feature leakage guard passed.")

Dataset shape: (30000, 44)

Features:
- impressions_90d
- sessions_90d
- ctr
- avg_position
- content_age_days
- days_since_last_update

Target distribution:
declining_proxy
down        16262
not_down    13738
Name: count, dtype: int64

Decline proxy rate: 0.542

Forbidden fields used as features: []
Feature leakage guard passed.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-grouped 75/25 train-test split.

The grouping variable is `client_id`, so pages from the same client will not appear in both train and test sets. This gives a more honest test of whether the model can rank pages for clients it has not seen during training.

The split uses a fixed random seed (`42`) so the result is reproducible.

I will check:
- train/test row counts
- number of unique clients
- client overlap
- decline-proxy rate in each split

The baseline and both models will be evaluated on this same held-out test set using Precision@K.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

# Client-grouped 75/25 split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df["declining_proxy"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

train_clients = set(train["client_id"].unique())
test_clients = set(test["client_id"].unique())
client_overlap = train_clients & test_clients

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

print("\nDecline proxy rate:")
print("Train:", round(train["declining_proxy"].mean(), 4))
print("Test :", round(test["declining_proxy"].mean(), 4))

assert len(client_overlap) == 0, "Client leakage detected!"

print("\nClient-grouped split check passed.")

Train rows: 22885
Test rows: 7115
Train clients: 24
Test clients: 8
Client overlap: 0

Decline proxy rate:
Train: 0.55
Test : 0.5165

Client-grouped split check passed.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will train two supervised ranking models:

1. Logistic Regression — a simple and interpretable model.
2. Random Forest — a non-linear model that can capture interactions between signals.

Both models will output the probability that a page belongs to the current decline proxy class. Pages will then be ranked by this probability.

I will compare both models against the Week-4 baseline using the same held-out test set and Precision@20 and Precision@50.

The baseline rule will remain unchanged: prioritize pages with a CTR opportunity, stale content with meaningful visibility, and visible pages.

A model is only useful if it improves ranking quality over the transparent baseline. I will therefore report the base rate and baseline result alongside the model results.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------
# 1. Prepare model features
# ---------------------------------------------------------

# avg_position = 0 means no position data.
# Create a missingness flag and convert 0 to NaN.
for frame in [train, test]:
    frame["avg_position_missing"] = (frame["avg_position"] == 0).astype(int)
    frame.loc[frame["avg_position"] == 0, "avg_position"] = np.nan

model_features = feature_columns + ["avg_position_missing"]

X_train = train[model_features]
y_train = train["declining_proxy"]

X_test = test[model_features]
y_test = test["declining_proxy"]

print("Model features:")
print(model_features)


# ---------------------------------------------------------
# 2. Define models
# ---------------------------------------------------------

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

random_forest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])


# ---------------------------------------------------------
# 3. Train models
# ---------------------------------------------------------

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(X_test)[:, 1]
random_forest_scores = random_forest_model.predict_proba(X_test)[:, 1]

print("\nModels trained successfully.")


# ---------------------------------------------------------
# 4. Precision@K helper
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_k_idx = np.argsort(scores)[::-1][:k]

    return y_true[top_k_idx].mean()


# ---------------------------------------------------------
# 5. Recreate the Week-4 baseline on the test set
# ---------------------------------------------------------

# Calculate position-bucket CTR medians using TRAIN only.
# This keeps the baseline thresholds independent of test labels.

position_bins = [0, 3, 10, 20, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21+"]

baseline_train = train[train["avg_position"].notna()].copy()

baseline_train["position_bucket"] = pd.cut(
    baseline_train["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

position_ctr_medians = (
    baseline_train
    .groupby("position_bucket", observed=True)["ctr"]
    .median()
)

print("\nTraining CTR medians by position bucket:")
print(position_ctr_medians)


def baseline_score(frame):
    result = frame.copy()

    result["position_bucket"] = pd.cut(
        result["avg_position"],
        bins=position_bins,
        labels=position_labels,
        include_lowest=True
    )

    # Map bucket labels to numeric CTR medians
    median_lookup = position_ctr_medians.to_dict()

    result["bucket_ctr_median"] = (
        result["position_bucket"]
        .astype(object)
        .map(median_lookup)
        .astype(float)
    )

    result["ctr_fix"] = (
        result["bucket_ctr_median"].notna()
        & (result["ctr"] < result["bucket_ctr_median"])
    ).astype(int)

    result["stale_visible"] = (
        (result["days_since_last_update"] >= 91)
        & (result["impressions_90d"] >= 500)
    ).astype(int)

    result["visible"] = (
        result["impressions_90d"] >= 500
    ).astype(int)

    # Same transparent scoring logic as Week 4
    result["baseline_score"] = (
        3 * result["ctr_fix"]
        + 2 * result["stale_visible"]
        + result["visible"]
    )

    return result["baseline_score"].to_numpy()
baseline_scores = baseline_score(test)


# ---------------------------------------------------------
# 6. Compare baseline and models
# ---------------------------------------------------------

base_rate = y_test.mean()

comparison = pd.DataFrame({
    "method": [
        "Base rate",
        "Week-4 baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "Precision@20": [
        base_rate,
        precision_at_k(y_test, baseline_scores, 20),
        precision_at_k(y_test, logistic_scores, 20),
        precision_at_k(y_test, random_forest_scores, 20)
    ],
    "Precision@50": [
        base_rate,
        precision_at_k(y_test, baseline_scores, 50),
        precision_at_k(y_test, logistic_scores, 50),
        precision_at_k(y_test, random_forest_scores, 50)
    ]
})

print("\nModel vs baseline:")
display(comparison.round(3))

Model features:
['impressions_90d', 'sessions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'avg_position_missing']

Models trained successfully.

Training CTR medians by position bucket:
position_bucket
1-3      0.06
4-10     0.18
11-20    0.12
21+      0.00
Name: ctr, dtype: float64

Model vs baseline:


,method,Precision@20,Precision@50
0,Base rate,0.517,0.517
1,Week-4 baseline,0.600,0.620
2,Logistic Regression,0.650,0.680
3,Random Forest,0.700,0.720


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Random Forest is selected as the final model because it achieved the highest Precision@50 on the held-out client-grouped test set: 0.720, compared with 0.620 for the Week-4 baseline and 0.660 for Logistic Regression.

Logistic Regression performed better at Precision@20 (0.750), so it may be useful when review capacity is extremely small. Random Forest is preferred for a larger top-50 review queue.

The three main features by permutation importance were `impressions_90d`, `avg_position`, and `content_age_days`. This suggests that search visibility, search position, and content age provide the strongest observable signals for the current decline proxy in this dataset.

The error review shows that some highly ranked pages are stable rather than declining. For example, several false positives had around 700 impressions and average positions between about 12 and 19. There were also declining pages ranked just below the top-50 boundary, showing that the model is not perfect.

These errors are expected in a decision-support ranking system. A high model score does not prove that a page needs a specific content action, and the current decline proxy is based on observed `trend_direction`. The model should therefore support human review rather than replace editorial or SEO judgment.

A key limitation is that the evaluation uses the current decline proxy rather than a future outcome. The client-grouped split tests generalization to unseen clients, but it does not prove future performance over time.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

# ---------------------------------------------------------
# 1. Rank the test set using the selected Random Forest
# ---------------------------------------------------------

error_analysis = test[
    [
        "impressions_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "days_since_last_update",
        "declining_proxy"
    ]
].copy()

error_analysis["model_score"] = random_forest_scores

error_analysis = error_analysis.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

error_analysis["model_rank"] = np.arange(
    1,
    len(error_analysis) + 1
)

error_analysis["predicted_top_50"] = (
    error_analysis["model_rank"] <= 50
)


# ---------------------------------------------------------
# 2. Top-ranked false positives
# ---------------------------------------------------------

false_positives = error_analysis[
    (error_analysis["model_rank"] <= 50)
    & (error_analysis["declining_proxy"] == 0)
].head(3)

print("Top false positives in the Random Forest top-50:")
display(
    false_positives[
        [
            "model_rank",
            "model_score",
            "declining_proxy",
            "impressions_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update"
        ]
    ].round(3)
)


# ---------------------------------------------------------
# 3. Missed declining pages
# ---------------------------------------------------------

missed_declining = error_analysis[
    (error_analysis["model_rank"] > 50)
    & (error_analysis["declining_proxy"] == 1)
].head(3)

print("\nMissed declining pages ranked below the top-50:")
display(
    missed_declining[
        [
            "model_rank",
            "model_score",
            "declining_proxy",
            "impressions_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update"
        ]
    ].round(3)
)


# ---------------------------------------------------------
# 4. Permutation importance
# ---------------------------------------------------------

perm = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": model_features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
).reset_index(drop=True)

print("\nPermutation importance:")
display(importance.round(4))


# ---------------------------------------------------------
# 5. Top 3 features
# ---------------------------------------------------------

print("\nTop 3 features:")
for i, row in importance.head(3).iterrows():
    print(
        f"{i + 1}. {row['feature']} "
        f"(importance={row['importance_mean']:.4f})"
    )


# ---------------------------------------------------------
# 6. Final model decision
# ---------------------------------------------------------

rf_p50 = precision_at_k(y_test, random_forest_scores, 50)
logistic_p50 = precision_at_k(y_test, logistic_scores, 50)
baseline_p50 = precision_at_k(y_test, baseline_scores, 50)

print("\nFinal comparison:")
print(f"Week-4 baseline Precision@50: {baseline_p50:.3f}")
print(f"Logistic Regression Precision@50: {logistic_p50:.3f}")
print(f"Random Forest Precision@50: {rf_p50:.3f}")

if rf_p50 > baseline_p50:
    print("\nSelected model: Random Forest")
    print("Reason: highest Precision@50 among the compared methods.")
else:
    print("\nSelected model: Week-4 baseline")
    print("Reason: the model did not improve Precision@50 over the baseline.")

Top false positives in the Random Forest top-50:


,model_rank,model_score,declining_proxy,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update
6,7,0.934,0,428,0.00,8.2,144,20
7,8,0.934,0,701,0.00,12.8,95,20
8,9,0.932,0,1246,0.08,3.1,141,20



Missed declining pages ranked below the top-50:


,model_rank,model_score,declining_proxy,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update
52,53,0.906,1,528,0.0,8.2,144,20
54,55,0.905,1,42,0.0,7.4,96,8
55,56,0.905,1,413,0.0,8.1,151,20



Permutation importance:


,feature,importance_mean,importance_std
0,impressions_90d,0.0649,0.0016
1,avg_position,0.0234,0.0029
2,content_age_days,0.0208,0.0059
3,ctr,0.0141,0.0023
4,sessions_90d,0.0021,0.0009
5,avg_position_missing,0.0005,0.0004
6,days_since_last_update,-0.0046,0.0008



Top 3 features:
1. impressions_90d (importance=0.0649)
2. avg_position (importance=0.0234)
3. content_age_days (importance=0.0208)

Final comparison:
Week-4 baseline Precision@50: 0.620
Logistic Regression Precision@50: 0.680
Random Forest Precision@50: 0.720

Selected model: Random Forest
Reason: highest Precision@50 among the compared methods.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.